# GNN3: FPL + GNN Residual Surrogate on IEEE 34-Bus

This notebook implements an FPL+GNN voltage surrogate for the IEEE 34-bus feeder using OpenDSS:

1. **Extract Z-bus / sensitivities**: build the load-bus admittance matrix $Y_{LL}$ from OpenDSS and compute $Z_{LL}=Y_{LL}^{-1}$ and the resulting linear voltage-magnitude sensitivity matrix $J$ around a chosen baseline.
2. **Generate dataset**: for many operating points (scenarios, timesteps), compute:
   - injections $x = [P_1,\dots,P_N,Q_1,\dots,Q_N]$,
   - true voltages $v^{\text{true}}$ from OpenDSS,
   - FPL prediction $v^{\text{fpl}} = v^{\text{base}} + J (x-x^{\text{base}})$,
   - residual label $\varepsilon = v^{\text{true}} - v^{\text{fpl}}$.
3. **Train FPL+GNN residual model**: a PFIdentity-style GNN that takes node / edge features and predicts $\widehat{\varepsilon}$, so that
   $$\widehat{v} = v^{\text{base}} + J (x-x^{\text{base}}) + g_\theta(\cdot).$$

We intentionally **reuse** the existing dataset-generation and GNN utilities (`run_injection_dataset.py`, `run_loadtype_dataset.py`, `run_deltav_dataset.py`, `run_gnn3_best7_train.py`) where possible.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

import opendssdirect as dss

import run_injection_dataset as inj
from run_loadtype_dataset import _compute_electrical_distance_from_source
from run_gnn3_best7_train import PFIdentityGNN

BASE_DIR = os.path.dirname(os.path.abspath("GNN3_FPL_GNN.ipynb"))
os.chdir(BASE_DIR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260303
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Extract Z-bus and Build Linear Sensitivity $J$

We build the full bus admittance matrix from OpenDSS, eliminate the slack bus to get $Y_{LL}$, invert it to $Z_{LL}$, and then approximate a **voltage-magnitude** sensitivity matrix $J$ around a chosen baseline operating point.

For simplicity we:
- use the existing IEEE 34 DSS model (`ieee34Mod1_with_loadshape.dss`),
- pick a **single baseline** scenario equal to `inj.BASELINE` at a reference time index `t0=0`,
- compute baseline complex voltages and magnitudes, and
- form a real-valued linear map from stacked injections $x=[P;Q]$ to magnitudes using a small finite-difference around the baseline.

This is not the most sophisticated FPL linearization, but it matches the structure
$$v^{\text{fpl}} = v^{\text{base}} + J (x - x^{\text{base}})$$
and plugs directly into the residual-learning pipeline.

In [ ]:
def build_zbus_and_baseline(t_index: int = 0, dP_kw: float = 1.0, dQ_kvar: float = 1.0):
    """Return (node_names, v_base_pu, x_base, J) for IEEE 34.

    - node_names: list of monitored bus.phase names in fixed order
    - v_base_pu: baseline voltage magnitudes [N]
    - x_base: baseline injections [2N] (P,Q per node, kW/kVAr, sign: +generation, -load)
    - J: finite-difference sensitivity [N x 2N] mapping (Δx) -> (Δv_mag)
    """
    # Compile DSS and set time
    dss_path = inj.compile_once()
    inj.setup_daily()
    inj.set_time_index(t_index)

    # Use baseline scenario from existing config
    baseline = dict(inj.BASELINE)
    P_load = baseline["P_load_total_kw"]
    Q_load = baseline["Q_load_total_kvar"]
    P_pv = baseline["P_pv_total_kw"]
    sigL = baseline["sigma_load"]
    sigPV = baseline["sigma_pv"]

    node_names, _, _, bus_to_phases = inj.get_all_bus_phase_nodes()
    N = len(node_names)
    node_to_idx = {n: i for i, n in enumerate(node_names)}

    loads_dss, dev_to_dss_load, dev_to_busph_load = inj.build_load_device_maps(bus_to_phases)
    pv_dss, pv_to_dss, pv_to_busph = inj.build_pv_device_maps()
    rng = np.random.default_rng(SEED)

    # Apply baseline snapshot (time-conditioned) and solve power flow
    _, busphP_load, busphQ_load, busphP_pv, busphQ_pv = inj.apply_snapshot_timeconditioned(
        P_load_total_kw=P_load,
        Q_load_total_kvar=Q_load,
        P_pv_total_kw=P_pv,
        mL_t=1.0,
        mPV_t=1.0,
        loads_dss=loads_dss,
        dev_to_dss_load=dev_to_dss_load,
        dev_to_busph_load=dev_to_busph_load,
        pv_dss=pv_dss,
        pv_to_dss=pv_to_dss,
        pv_to_busph=pv_to_busph,
        sigma_load=sigL,
        sigma_pv=sigPV,
        rng=rng,
    )
    dss.Solution.Solve()
    if not dss.Solution.Converged():
        raise RuntimeError("Baseline PF did not converge.")

    # Baseline voltages (magnitudes) and baseline injections per node
    vmag_base, _ = inj.get_all_node_voltage_pu_and_angle_filtered(node_names)
    v_base = np.asarray(vmag_base, dtype=np.float64)

    P_node = np.zeros(N, dtype=np.float64)
    Q_node = np.zeros(N, dtype=np.float64)
    for i, n in enumerate(node_names):
        bus, phs = n.split(".")
        ph = int(phs)
        p_load = float(busphP_load.get((bus, ph), 0.0))
        q_load = float(busphQ_load.get((bus, ph), 0.0))
        p_pv = float(busphP_pv.get((bus, ph), 0.0))
        q_pv = float(busphQ_pv.get((bus, ph), 0.0))
        P_node[i] = p_pv - p_load
        Q_node[i] = q_pv - q_load

    x_base = np.concatenate([P_node, Q_node], axis=0)  # [2N]

    # Finite-difference sensitivities J ≈ ∂v/∂x at baseline
    J = np.zeros((N, 2 * N), dtype=np.float64)

    def _solve_with_perturb(idx_node, dP, dQ):
        # Re-compile and reset snapshot to get a clean base each time
        inj.dss.Basic.ClearAll()
        dss.Text.Command(f'compile "{dss_path}"')
        inj._apply_voltage_bases()
        inj.setup_daily()
        inj.set_time_index(t_index)

        node_names_s, _, _, bus_to_phases_s = inj.get_all_bus_phase_nodes()
        assert node_names_s == node_names
        loads_dss_s, dev_to_dss_load_s, dev_to_busph_load_s = inj.build_load_device_maps(bus_to_phases_s)
        pv_dss_s, pv_to_dss_s, pv_to_busph_s = inj.build_pv_device_maps()
        rng_s = np.random.default_rng(SEED)
        # baseline again
        _, busphP_load_s, busphQ_load_s, busphP_pv_s, busphQ_pv_s = inj.apply_snapshot_timeconditioned(
            P_load_total_kw=P_load,
            Q_load_total_kvar=Q_load,
            P_pv_total_kw=P_pv,
            mL_t=1.0,
            mPV_t=1.0,
            loads_dss=loads_dss_s,
            dev_to_dss_load=dev_to_dss_load_s,
            dev_to_busph_load=dev_to_busph_load_s,
            pv_dss=pv_dss_s,
            pv_to_dss=pv_to_dss_s,
            pv_to_busph=pv_to_busph_s,
            sigma_load=sigL,
            sigma_pv=sigPV,
            rng=rng_s,
        )
        # Apply small perturbation at selected node as extra injection
        node = node_names[idx_node]
        bus, phs = node.split(".")
        ph = int(phs)
        # Implemented as an extra constant-PQ load of -dP - j dQ at that node
        dss.Loads.New(name="fpl_sens_load", bus=f"{bus}.{ph}", phases=1, conn="Wye",
                     model=1, kv=4.16, kW=-dP, kvar=-dQ)

        dss.Solution.Solve()
        if not dss.Solution.Converged():
            return None
        vmag_new, _ = inj.get_all_node_voltage_pu_and_angle_filtered(node_names)
        return np.asarray(vmag_new, dtype=np.float64)

    for i in range(N):
        # ∂v/∂P_i
        v_plus = _solve_with_perturb(i, dP_kw, 0.0)
        if v_plus is not None:
            J[:, i] = (v_plus - v_base) / dP_kw
        # ∂v/∂Q_i
        v_plus = _solve_with_perturb(i, 0.0, dQ_kvar)
        if v_plus is not None:
            J[:, N + i] = (v_plus - v_base) / dQ_kvar

    return node_names, v_base, x_base, J

## 2. Dataset Generation: FPL + Residual Labels

We now generate a dataset that mirrors the existing `loadtype` / `delta-V` structure but with **residual labels**:

- Features per node: reuse the **load-type + electrical distance** features from `gnn_samples_loadtype_full`.
- Target per node: residual
  $$\varepsilon_i = v^{\text{true}}_i - v^{\text{base}}_i - (J (x-x^{\text{base}}))_i.$$

This gives a dataset suitable for training a PFIdentityGNN to learn $g_\theta$.

In [ ]:
def generate_fpl_residual_dataset(
    out_dir: str = "gnn_samples_fpl_residual_full",
    n_scenarios: int = 200,
    k_snapshots_per_scenario_total: int = 960,
    master_seed: int = 20260303,
):
    os.makedirs(out_dir, exist_ok=True)
    edge_csv = os.path.join(out_dir, "gnn_edges_phase_static.csv")
    node_csv = os.path.join(out_dir, "gnn_node_features_and_targets.csv")
    sample_csv = os.path.join(out_dir, "gnn_sample_meta.csv")
    node_index_csv = os.path.join(out_dir, "gnn_node_index_master.csv")

    # Baseline linearization
    node_names, v_base, x_base, J = build_zbus_and_baseline(t_index=0)
    N = len(node_names)
    node_to_idx = {n: i for i, n in enumerate(node_names)}

    # Save master node index
    pd.DataFrame({"node": node_names, "node_idx": np.arange(N, dtype=int)}).to_csv(
        node_index_csv, index=False
    )

    # Reuse static phase edges and electrical distance
    inj.extract_static_phase_edges_to_csv(node_names_master=node_names, edge_csv_path=edge_csv)
    node_to_electrical_dist = _compute_electrical_distance_from_source(node_names, edge_csv)

    # Load shapes for time sampling
    dss_path = inj.compile_once()
    inj.setup_daily()
    csvL_token, _ = inj.find_loadshape_csv_in_dss(dss_path, "5minDayShape")
    csvPV_token, _ = inj.find_loadshape_csv_in_dss(dss_path, "IrradShape")
    csvL = inj.resolve_csv_path(csvL_token, dss_path)
    csvPV = inj.resolve_csv_path(csvPV_token, dss_path)
    mL = inj.read_profile_csv_two_col_noheader(csvL, npts=inj.NPTS, debug=False)
    mPV = inj.read_profile_csv_two_col_noheader(csvPV, npts=inj.NPTS, debug=False)

    rng_master = np.random.default_rng(master_seed)
    rows_sample, rows_node = [], []
    sample_id = 0
    kept = skipped_nonconv = skipped_badV = 0

    for s in range(n_scenarios):
        inj.dss.Basic.ClearAll()
        inj.dss.Text.Command(f'compile "{dss_path}"')
        inj._apply_voltage_bases()
        inj.setup_daily()

        node_names_s, _, _, bus_to_phases = inj.get_all_bus_phase_nodes()
        if node_names_s != node_names:
            raise RuntimeError("Node list changed across scenarios.")

        loads_dss, dev_to_dss_load, dev_to_busph_load = inj.build_load_device_maps(bus_to_phases)
        pv_dss, pv_to_dss, pv_to_busph = inj.build_pv_device_maps()

        sc = inj.sample_scenario_from_baseline(inj.BASELINE, inj.RANGES, rng_master)
        P_load = sc["P_load_total_kw"]
        Q_load = sc["Q_load_total_kvar"]
        P_pv = sc["P_pv_total_kw"]
        sigL = sc["sigma_load"]
        sigPV = sc["sigma_pv"]

        prof_load = mL
        prof_pv = mPV
        prof_net = (P_load * mL) - (P_pv * mPV)

        rng_times = np.random.default_rng(int(rng_master.integers(0, 2**31 - 1)))
        times = inj.select_times_three_profiles(
            prof_load=prof_load,
            prof_pv=prof_pv,
            prof_net=prof_net,
            K_total=k_snapshots_per_scenario_total,
            bins_by_profile={"load": 10, "pv": 10, "net": 10},
            include_anchors=True,
            rng=rng_times,
        )
        times = [int(t) for t in times]
        rng_solve = np.random.default_rng(int(rng_master.integers(0, 2**31 - 1)))

        for t in times:
            inj.set_time_index(t)
            totals, busphP_load, busphQ_load, busphP_pv, busphQ_pv = inj.apply_snapshot_timeconditioned(
                P_load_total_kw=P_load,
                Q_load_total_kvar=Q_load,
                P_pv_total_kw=P_pv,
                mL_t=float(mL[t]),
                mPV_t=float(mPV[t]),
                loads_dss=loads_dss,
                dev_to_dss_load=dev_to_dss_load,
                dev_to_busph_load=dev_to_busph_load,
                pv_dss=pv_dss,
                pv_to_dss=pv_to_dss,
                pv_to_busph=pv_to_busph,
                sigma_load=sigL,
                sigma_pv=sigPV,
                rng=rng_solve,
            )

            inj.dss.Solution.Solve()
            if not inj.dss.Solution.Converged():
                skipped_nonconv += 1
                continue

            vmag_m, vang_m = inj.get_all_node_voltage_pu_and_angle_filtered(node_names)
            vmag_arr = np.asarray(vmag_m, dtype=np.float64)
            if (
                (not np.isfinite(vmag_arr).all())
                or (vmag_arr.min() < inj.VMAG_PU_MIN)
                or (vmag_arr.max() > inj.VMAG_PU_MAX)
            ):
                skipped_badV += 1
                continue

            # Build nodal injections P,Q at this snapshot
            P_node = np.zeros(N, dtype=np.float64)
            Q_node = np.zeros(N, dtype=np.float64)
            for i, n in enumerate(node_names):
                bus, phs = n.split(".")
                ph = int(phs)
                p_load = float(busphP_load.get((bus, ph), 0.0))
                q_load = float(busphQ_load.get((bus, ph), 0.0))
                p_pv = float(busphP_pv.get((bus, ph), 0.0))
                q_pv = float(busphQ_pv.get((bus, ph), 0.0))
                P_node[i] = p_pv - p_load
                Q_node[i] = q_pv - q_load

            x = np.concatenate([P_node, Q_node], axis=0)
            dx = x - x_base
            v_fpl = v_base + J @ dx
            eps = vmag_arr - v_fpl  # residual label

            rows_sample.append(
                {
                    "sample_id": sample_id,
                    "scenario_id": s,
                    "t_index": t,
                    "t_minutes": t * inj.STEP_MIN,
                }
            )

            for i, n in enumerate(node_names):
                bus, phs = n.split(".")
                ph = int(phs)
                elec_dist = float(node_to_electrical_dist.get(n, 0.0))
                vm, va = float(vmag_arr[i]), float(vang_m[i])

                rows_node.append(
                    {
                        "sample_id": sample_id,
                        "node": n,
                        "node_idx": int(node_to_idx[n]),
                        "bus": bus,
                        "phase": int(ph),
                        "electrical_distance_ohm": elec_dist,
                        "P_node_kw": float(P_node[i]),
                        "Q_node_kvar": float(Q_node[i]),
                        "vmag_pu_true": vm,
                        "vmag_pu_fpl": float(v_fpl[i]),
                        "vmag_pu_resid": float(eps[i]),
                    }
                )

            sample_id += 1
            kept += 1

        print(
            f"[scenario {s+1}/{n_scenarios}] kept={kept} skip_nonconv={skipped_nonconv} "
            f"skip_badV={skipped_badV} Pload={P_load:.1f} Qload={Q_load:.1f} Ppv={P_pv:.1f}"
        )

    df_sample = pd.DataFrame(rows_sample)
    df_node = pd.DataFrame(rows_node)
    df_sample.to_csv(sample_csv, index=False)
    df_node.to_csv(node_csv, index=False)

    print(f"\n[FPL+RESIDUAL DATASET] Saved to {out_dir}/")
    print(f"  {node_csv} | samples={df_sample['sample_id'].nunique()} | node-rows={len(df_node)}")
    return df_sample, df_node, edge_csv

## 3. Train FPL+GNN Residual Model

We now train a PFIdentityGNN on the residual labels `vmag_pu_resid`, using node features:

- `electrical_distance_ohm`
- local injections `P_node_kw`, `Q_node_kvar`

The model learns $g_\theta$ so that at inference time we can form
$$\widehat{v} = v^{\text{base}} + J (x-x^{\text{base}}) + g_\theta(\cdot).$$

In [ ]:
def train_fpl_gnn(
    out_dir: str = "gnn_samples_fpl_residual_full",
    h_dim: int = 96,
    num_layers: int = 3,
    batch_size: int = 64,
    epochs: int = 50,
    test_frac: float = 0.2,
):
    edge_csv = os.path.join(out_dir, "gnn_edges_phase_static.csv")
    node_csv = os.path.join(out_dir, "gnn_node_features_and_targets.csv")
    df_e = pd.read_csv(edge_csv)
    df_n = pd.read_csv(node_csv)

    feature_cols = ["electrical_distance_ohm", "P_node_kw", "Q_node_kvar"]
    target_col = "vmag_pu_resid"

    required = {"sample_id", "node_idx", target_col} | set(feature_cols)
    if required - set(df_n.columns):
        raise RuntimeError(f"Missing columns: {required - set(df_n.columns)}")

    for c in ["u_idx", "v_idx", "R_full", "X_full"]:
        df_e[c] = pd.to_numeric(df_e[c], errors="coerce")
    for c in feature_cols + [target_col, "sample_id", "node_idx"]:
        df_n[c] = pd.to_numeric(df_n[c], errors="coerce")

    df_e = df_e.replace([np.inf, -np.inf], np.nan).dropna(subset=["u_idx", "v_idx", "R_full", "X_full"]).reset_index(drop=True)
    df_n = df_n.replace([np.inf, -np.inf], np.nan).dropna(subset=list(required)).reset_index(drop=True)

    df_e["edge_id"] = np.arange(len(df_e), dtype=int)
    E = int(len(df_e))
    N = int(df_n["node_idx"].max()) + 1

    df_n = df_n.sort_values(["sample_id", "node_idx"]).reset_index(drop=True)
    counts = df_n.groupby("sample_id")["node_idx"].count()
    good_ids = counts[counts == N].index.to_numpy()
    df_n = df_n[df_n["sample_id"].isin(good_ids)].copy().sort_values(["sample_id", "node_idx"]).reset_index(drop=True)

    all_ids = df_n["sample_id"].unique()
    S = len(all_ids)

    X_all = df_n[feature_cols].to_numpy(dtype=np.float32).reshape(S, N, -1)
    Y_all = df_n[target_col].to_numpy(dtype=np.float32).reshape(S, N, 1)

+    node_in_dim = X_all.shape[-1]
    edge_index = torch.tensor(df_e[["u_idx", "v_idx"]].to_numpy().T, dtype=torch.long)
    edge_attr = torch.tensor(df_e[["R_full", "X_full"]].to_numpy(), dtype=torch.float32)
    edge_id = torch.tensor(df_e["edge_id"].to_numpy(), dtype=torch.long)

    rng = np.random.default_rng(SEED)
    perm = rng.permutation(S)
    n_test = int(np.floor(test_frac * S))
    test_idx, train_idx = perm[:n_test], perm[n_test:]

    def make_ds(idx):
        data_list = []
        for k in idx:
            x = torch.tensor(X_all[k], dtype=torch.float32)
            y = torch.tensor(Y_all[k], dtype=torch.float32)
            g = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_attr, edge_id=edge_id, num_nodes=N)
            data_list.append(g)
        return data_list

    train_loader = DataLoader(make_ds(train_idx), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(make_ds(test_idx), batch_size=batch_size, shuffle=False)

    model = PFIdentityGNN(
        num_nodes=N,
        num_edges=E,
        node_in_dim=node_in_dim,
        edge_in_dim=2,
        out_dim=1,
        node_emb_dim=16,
        edge_emb_dim=8,
        h_dim=h_dim,
        num_layers=num_layers,
        use_norm=False,
    ).to(DEVICE)

    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    def compute_metrics(yhat, ytrue):
        err = (yhat - ytrue).squeeze(-1)
        mae = err.abs().mean()
        rmse = torch.sqrt((err ** 2).mean())
        return mae, rmse

    @torch.no_grad()
    def evaluate(loader):
        model.eval()
        mae_sum = torch.tensor(0.0, device=DEVICE)
        rmse_sum = torch.tensor(0.0, device=DEVICE)
        n_batches = 0
        for data in loader:
            data = data.to(DEVICE)
            mae, rmse = compute_metrics(model(data), data.y)
            mae_sum += mae
            rmse_sum += rmse
            n_batches += 1
        return (mae_sum / max(1, n_batches)).item(), (rmse_sum / max(1, n_batches)).item()

    best_rmse = float("inf")
    best_state = None
    patience = 10
    min_delta = 1e-6

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        nb = 0
        for data in train_loader:
            data = data.to(DEVICE)
            opt.zero_grad()
            loss = F.mse_loss(model(data), data.y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()
            total_loss += float(loss.item())
            nb += 1
        mae_t, rmse_t = evaluate(test_loader)
        if (best_rmse - rmse_t) > min_delta:
            best_rmse = rmse_t
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 10
        else:
            patience -= 1
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | RMSE={rmse_t:.6f} | best={best_rmse:.6f} | patience={patience}")
        if patience <= 0:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    mae_f, rmse_f = evaluate(test_loader)
    print(f"Final: MAE={mae_f:.6f} RMSE={rmse_f:.6f}")
    return model


# Example (run manually in the notebook):
# df_s, df_n, edge_path = generate_fpl_residual_dataset()
# model_fpl_gnn = train_fpl_gnn()